# Notebook 02 — Traffic Forecasting

**Goal:** Train and evaluate two spatiotemporal GNN architectures on METR-LA (and PEMS-D3/D4/D7 if available):
1. **GCN-GRU** (baseline — differentiable graph convolution + GRU)
2. **AdaptiveNAS-GNN** (ours — DARTS-style NAS with learned graph operations)

**Evaluation:** MAE / RMSE / MAPE at 15 min, 30 min, 60 min forecast horizons.

> All cells run automatically. Two-phase NAS training: architecture search, then retrain with best ops.

In [ ]:
# Auto-detect repo root for JupyterLab / GCP / local execution

from pathlib import Path

import sys



def find_repo_root(start=None):

    start_path = Path(start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:

        if (candidate / 'src').exists() and (candidate / 'notebooks').exists():

            return candidate

    return start_path



REPO_ROOT = find_repo_root()

DATA_ROOT = REPO_ROOT / 'data'

CKPT_ROOT = REPO_ROOT / 'checkpoints'

EXPS_ROOT = REPO_ROOT / 'experiments'

EXPS_ROOT.mkdir(exist_ok=True)

CKPT_ROOT.mkdir(exist_ok=True)

sys.path.insert(0, str(REPO_ROOT))



import torch

import torch.optim as optim

import numpy as np

import matplotlib.pyplot as plt



from src.data_loaders import load_metr_la, load_pems

from src.models import GCNGRU, AdaptiveNASGNN

from src.training import ForecastingTrainer

from src.evaluation import evaluate_forecasting



DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device: {DEVICE}')

if torch.cuda.is_available():

    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Cell 2: Training configuration ───────────────────────────────────────────
CFG = {
    'seq_in'         : 12,     # 1 hour history (5-min intervals)
    'seq_out'        : 12,     # 1 hour ahead   (5-min intervals)
    'batch_size'     : 64,
    'workers'        : 2,
    'hidden_dim'     : 64,
    'num_layers'     : 2,
    'epochs_baseline': 100,
    'epochs_search'  : 30,    # NAS architecture search phase
    'epochs_retrain' : 100,   # NAS retrain phase
    'lr'             : 1e-3,
    'lr_arch'        : 3e-4,   # architecture param lr (NAS)
    'weight_decay'   : 1e-4,
    'patience'       : 15,
}

if DEVICE == 'cpu':
    CFG['batch_size'] = 16
    CFG['epochs_baseline'] = 3
    CFG['epochs_search']   = 2
    CFG['epochs_retrain']  = 3
    print('CPU mode: reduced epochs for demo run')

print('Configuration:', CFG)

In [ ]:
# ── Cell 3: Load METR-LA dataset ─────────────────────────────────────────────
metr_h5   = DATA_ROOT / 'metr-la' / 'metr-la.h5'
metr_pkl  = DATA_ROOT / 'metr-la' / 'adj_mx.pkl'

if not metr_h5.exists():
    raise FileNotFoundError(f'METR-LA HDF5 not found: {metr_h5}\nPlease copy metr-la.h5 into data/metr-la/')

print('Loading METR-LA...')
train_loader, val_loader, test_loader, scaler, adj, num_nodes = load_metr_la(
    h5_path=str(metr_h5),
    adj_path=str(metr_pkl) if metr_pkl.exists() else None,
    seq_in=CFG['seq_in'],
    seq_out=CFG['seq_out'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['workers'],
)

print(f'Nodes      : {num_nodes}')
print(f'Train batches: {len(train_loader)}  |  Val: {len(val_loader)}  |  Test: {len(test_loader)}')

x, y = next(iter(train_loader))
print(f'Batch x: {tuple(x.shape)}  →  y: {tuple(y.shape)}')

In [ ]:
# ── Cell 4: Explore adjacency matrix and speed data ───────────────────────────
import torch.nn.functional as F

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Adjacency matrix heatmap (show first 50×50 for clarity)
adj_np = adj.cpu().numpy() if isinstance(adj, torch.Tensor) else np.array(adj)
axes[0].imshow(adj_np[:50, :50], cmap='Blues', vmin=0, vmax=1)
axes[0].set_title('Adjacency Matrix (first 50 nodes)')
axes[0].set_xlabel('Node'); axes[0].set_ylabel('Node')
plt.colorbar(axes[0].images[0], ax=axes[0])

# Speed time series for 5 sample nodes
raw_x, _ = next(iter(test_loader))
sample = raw_x[0, :, :5, 0].numpy()   # [T, 5 nodes, speed]
for i in range(5):
    axes[1].plot(sample[:, i], label=f'Node {i}')
axes[1].set_title('Speed Time Series (5 sample nodes)')
axes[1].set_xlabel('Timestep (5-min interval)')
axes[1].set_ylabel('Speed (normalised)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(EXPS_ROOT / 'forecasting_metrla_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 5: Train GCN-GRU baseline ───────────────────────────────────────────
print('=' * 60)
print('TRAINING GCN-GRU BASELINE')
print('=' * 60)

model_gcn = GCNGRU(
    num_nodes=num_nodes,
    in_features=x.shape[-1],   # typically 2 (speed + time-of-day)
    hidden_dim=CFG['hidden_dim'],
    num_layers=CFG['num_layers'],
    seq_out=CFG['seq_out'],
).to(DEVICE)

print(f'GCN-GRU parameters: {sum(p.numel() for p in model_gcn.parameters()):,}')

opt_gcn = optim.Adam(model_gcn.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_gcn = optim.lr_scheduler.MultiStepLR(opt_gcn, milestones=[50, 80], gamma=0.5)

trainer_gcn = ForecastingTrainer(
    model=model_gcn, optimizer=opt_gcn, scheduler=sched_gcn,
    adj=adj, scaler=scaler, device=DEVICE,
    experiment_name='gcn_gru_metrla',
    save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
)
trainer_gcn.load_checkpoint('last.pt')

history_gcn = trainer_gcn.train(
    train_loader, val_loader,
    epochs=CFG['epochs_baseline'], patience=CFG['patience'],
    metric_key='mae',
)

In [ ]:
# ── Cell 6: Evaluate GCN-GRU ─────────────────────────────────────────────────
trainer_gcn.load_checkpoint('best.pt')
gcn_results = evaluate_forecasting(
    model_gcn, test_loader, scaler, adj, DEVICE,
    horizons=[3, 6, 12]  # 15, 30, 60 min
)

print('\nGCN-GRU Results on METR-LA:')
for horizon, metrics in gcn_results.items():
    print(f'  {horizon}  →  MAE: {metrics["mae"]:.4f}  RMSE: {metrics["rmse"]:.4f}  MAPE: {metrics["mape"]:.2f}%')

In [ ]:
# ── Cell 7: AdaptiveNAS-GNN — Phase 1: Architecture Search ────────────────────
print('=' * 60)
print('AdaptiveNAS-GNN — PHASE 1: Architecture Search')
print('=' * 60)

model_nas = AdaptiveNASGNN(
    num_nodes=num_nodes,
    in_features=x.shape[-1],
    hidden_dim=CFG['hidden_dim'],
    num_layers=CFG['num_layers'],
    seq_out=CFG['seq_out'],
).to(DEVICE)

print(f'AdaptiveNAS-GNN parameters: {sum(p.numel() for p in model_nas.parameters()):,}')
print(f'Architecture parameters  : {sum(p.numel() for p in model_nas.arch_parameters()):,}')

# Bilevel: separate optimizers for model weights and architecture params
opt_nas_model = optim.Adam(model_nas.model_parameters(), lr=CFG['lr'],      weight_decay=CFG['weight_decay'])
opt_nas_arch  = optim.Adam(model_nas.arch_parameters(),  lr=CFG['lr_arch'], weight_decay=1e-3)

trainer_nas_search = ForecastingTrainer(
    model=model_nas,
    optimizer=opt_nas_model,
    arch_optimizer=opt_nas_arch,
    adj=adj, scaler=scaler, device=DEVICE,
    experiment_name='nas_gnn_search',
    save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
)
trainer_nas_search.load_checkpoint('last.pt')

history_search = trainer_nas_search.train(
    train_loader, val_loader,
    epochs=CFG['epochs_search'], patience=CFG['patience'],
    metric_key='mae',
)

discovered_arch = model_nas.get_architecture()
print('\nDiscovered Architecture:')
for layer_i, ops in enumerate(discovered_arch):
    print(f'  Layer {layer_i}: graph_op={ops["graph_op"]}, dilation={ops["dilation"]}, activation={ops["activation"]}')

In [ ]:
# ── Cell 8: AdaptiveNAS-GNN — Phase 2: Retrain with Fixed Architecture ────────
print('=' * 60)
print('AdaptiveNAS-GNN — PHASE 2: Retrain')
print('=' * 60)

model_nas.discretize()   # freezes arch params, uses argmax ops

opt_retrain   = optim.Adam(model_nas.model_parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_retrain = optim.lr_scheduler.CosineAnnealingLR(opt_retrain, T_max=CFG['epochs_retrain'], eta_min=1e-6)

trainer_nas_retrain = ForecastingTrainer(
    model=model_nas, optimizer=opt_retrain, scheduler=sched_retrain,
    adj=adj, scaler=scaler, device=DEVICE,
    experiment_name='nas_gnn_retrain',
    save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
)

history_retrain = trainer_nas_retrain.train(
    train_loader, val_loader,
    epochs=CFG['epochs_retrain'], patience=CFG['patience'],
    metric_key='mae',
)

In [ ]:
# ── Cell 9: Evaluate AdaptiveNAS-GNN ─────────────────────────────────────────
trainer_nas_retrain.load_checkpoint('best.pt')
nas_results = evaluate_forecasting(
    model_nas, test_loader, scaler, adj, DEVICE,
    horizons=[3, 6, 12]
)

print('\nAdaptiveNAS-GNN Results on METR-LA:')
for horizon, metrics in nas_results.items():
    print(f'  {horizon}  →  MAE: {metrics["mae"]:.4f}  RMSE: {metrics["rmse"]:.4f}  MAPE: {metrics["mape"]:.2f}%')

In [ ]:
# ── Cell 10: Comparison table ─────────────────────────────────────────────────
from src.utils import make_results_table

horizons_label = {'h3': '15 min', 'h6': '30 min', 'h12': '60 min'}

rows = {}
for model_name, res in [('GCN-GRU (baseline)', gcn_results), ('AdaptiveNAS-GNN (ours)', nas_results)]:
    row = {}
    for h_key, h_label in horizons_label.items():
        h_data = res.get(h_key, {})
        row[f'{h_label}_MAE']  = h_data.get('mae', float('nan'))
        row[f'{h_label}_RMSE'] = h_data.get('rmse', float('nan'))
        row[f'{h_label}_MAPE'] = h_data.get('mape', float('nan'))
    rows[model_name] = row

table = make_results_table(rows)
print('\n=== METR-LA Forecasting Results ===')
print(table)

(EXPS_ROOT / 'forecasting_metrla_results.md').write_text(
    '# METR-LA Forecasting Results\n\n' + table
)
print('Saved results to experiments/forecasting_metrla_results.md')

In [ ]:
# ── Cell 11: Prediction visualisation ────────────────────────────────────────
model_gcn.eval()
model_nas.eval()

TEST_NODES = [0, 5, 10]   # nodes to visualise

batch_x, batch_y = next(iter(test_loader))
batch_x_dev = batch_x.to(DEVICE)

with torch.no_grad():
    if isinstance(adj, torch.Tensor):
        adj_dev = adj.to(DEVICE)
    else:
        adj_dev = torch.tensor(adj, dtype=torch.float32, device=DEVICE)

    pred_gcn_raw = model_gcn(batch_x_dev, adj_dev)   # [B, T_out, N, 1]
    pred_nas_raw = model_nas(batch_x_dev, adj_dev)

# Inverse-transform
def inv(t):
    return scaler.inverse_transform(t.cpu().numpy()[..., 0])  # [B, T, N]

gt  = inv(batch_y)           # [B, T_out, N]
p_gcn = inv(pred_gcn_raw)
p_nas = inv(pred_nas_raw)

fig, axes = plt.subplots(len(TEST_NODES), 1, figsize=(12, 4 * len(TEST_NODES)), sharex=True)
if len(TEST_NODES) == 1:
    axes = [axes]

for ax, node_i in zip(axes, TEST_NODES):
    t = np.arange(CFG['seq_out'])
    ax.plot(t, gt[0, :, node_i],    'k-',  label='Ground Truth', linewidth=2)
    ax.plot(t, p_gcn[0, :, node_i], 'b--', label='GCN-GRU')
    ax.plot(t, p_nas[0, :, node_i], 'r-',  label='AdaptiveNAS-GNN')
    ax.set_ylabel('Speed (mph)')
    ax.set_title(f'Node {node_i}')
    ax.legend(); ax.grid(True, alpha=0.3)

axes[-1].set_xlabel('Forecast Horizon (× 5 min)')
plt.suptitle('METR-LA Speed Forecasting Sample', fontsize=13)
plt.tight_layout()
plt.savefig(EXPS_ROOT / 'forecasting_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 12: Optional — PEMS datasets ────────────────────────────────────────
pems_datasets = {
    'PEMS-D3': (DATA_ROOT / 'metr-la' / 'd3_speed.npz', DATA_ROOT / 'metr-la' / 'd3_adj.csv'),
    'PEMS-D4': (DATA_ROOT / 'metr-la' / 'd4_speed.npz', DATA_ROOT / 'metr-la' / 'd4_adj.csv'),
    'PEMS-D7': (DATA_ROOT / 'metr-la' / 'd7_speed.npz', DATA_ROOT / 'metr-la' / 'd7_adj.csv'),
}

for pems_name, (npz_path, csv_path) in pems_datasets.items():
    if not npz_path.exists():
        print(f'{pems_name}: not found at {npz_path}, skipping.')
        continue

    print(f'\nLoading {pems_name}...')
    tr_l, va_l, te_l, sc, adj_p, nn = load_pems(
        npz_path=str(npz_path),
        csv_path=str(csv_path) if csv_path.exists() else None,
        seq_in=CFG['seq_in'], seq_out=CFG['seq_out'],
        batch_size=CFG['batch_size'], num_workers=CFG['workers'],
    )
    print(f'  Nodes: {nn} | Train: {len(tr_l.dataset)} | Test: {len(te_l.dataset)}')

    m_p = AdaptiveNASGNN(
        num_nodes=nn, in_features=CFG['seq_in'], hidden_dim=CFG['hidden_dim'],
        num_layers=CFG['num_layers'], seq_out=CFG['seq_out'],
    ).to(DEVICE)

    opt_p = optim.Adam(m_p.model_parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    opt_a = optim.Adam(m_p.arch_parameters(), lr=CFG['lr_arch'])

    t_p = ForecastingTrainer(
        model=m_p, optimizer=opt_p, arch_optimizer=opt_a,
        adj=adj_p, scaler=sc, device=DEVICE,
        experiment_name=f'nas_{pems_name.lower().replace("-","_")}',
        save_dir=str(CKPT_ROOT), log_dir=str(REPO_ROOT / 'runs'),
    )
    t_p.train(tr_l, va_l, epochs=CFG['epochs_retrain'], patience=CFG['patience'])
    t_p.load_checkpoint('best.pt')

    res_p = evaluate_forecasting(m_p, te_l, sc, adj_p, DEVICE)
    print(f'  {pems_name} @60-min → MAE: {res_p.get("h12", {}).get("mae", float("nan")):.4f}')
    
print('\nPEMS evaluation complete.')

---
## Traffic Forecasting Complete!

Results saved to `experiments/`. Continue with **03_anomaly_detection.ipynb**.